# Test ai for R script

In [ ]:
%%ai
write a R script example of species distribution model in the marine realm

library(sdm)
library(raster)
library(dismo)

# Load marine environmental data (e.g., WorldClim or Bio-ORACLE)
# Assuming 'env_layers' is a RasterStack of marine variables (SST, Salinity, etc.)
# Assuming 'occurrence_data' is a data.frame with columns: species, lon, lat

# Prepare data
# QPan: sdmData: https://www.rdocumentation.org/packages/sdm/versions/1.2-59/topics/sdmData
d <- sdmData(formula = species ~ ., train = occurrence_data, predictors = env_layers)

# Run model
m <- sdm(species ~ ., data = d, methods = c('glm', 'rf', 'maxent'))

# Predict distribution
p <- predict(m, env_layers, filename = 'prediction.img')

# Plot results
plot(p)# Evaluate model performance
eval <- getEvaluation(m, stat = c('AUC', 'TSS'))
print(eval)

# Ensemble prediction
p_ensemble <- ensemble(m, env_layers, filename = 'ensemble_prediction.img', setting = list(method = 'weighted', stat = 'AUC'))
plot(p_ensemble)


In [ ]:
%%ai
write R code to explain cubing engine in biodiversity domain, include GBIF datasets.

library(rgbif)
library(gdalcubes)
library(stars)

# Fetch occurrence data from GBIF
occ_data <- occ_data(scientificName = "Gadus morhua", limit = 1000, hasCoordinate = TRUE)
df <- occ_data$data[, c("decimalLongitude", "decimalLatitude", "eventDate")]
colnames(df) <- c("x", "y", "t")

# Define a cube view
v <- cube_view(extent = list(t0 = "2020-01-01", t1 = "2020-12-31", 
                             left = -20, right = 20, bottom = 40, top = 70),
               nx = 100, ny = 100, dt = "P1M", projection = "EPSG:4326")

# Create a data cube from occurrences
cube <- rasterize_cube(df, v, edge_size = 1)

# Aggregate occurrences by cell
cube_agg <- cube_apply(cube, function(x) sum(x, na.rm = TRUE))

# Plot the biodiversity cube
plot(cube_agg)# Calculate temporal trends
trend <- cube_apply(cube, function(x) {
  if (all(is.na(x))) return(NA)
  return(sum(x, na.rm = TRUE))
})

# Convert to stars object for further analysis
st_cube <- st_as_stars(cube_agg)
plot(st_cube, main = "Gadus morhua occurrence density (2020)")

# Export to NetCDF
write_ncdf(cube_agg, "gadus_morhua_cube.nc")

# Summary statistics of the cube
summary_stats <- aggregate(st_cube, "time", sum)
print(summary_stats)

# Visualize temporal change
plot(summary_stats, main = "Monthly occurrence counts")

# Calculate mean occurrence density across the study period
mean_density <- st_apply(st_cube, c("x", "y"), mean)
plot(mean_density, main = "Mean monthly occurrence density")

# Extract time series for a specific location
ts_location <- st_extract(st_cube, st_point(c(0, 55)))
plot(ts_location, main = "Temporal trend at 0E, 55N")
                       
# Clean up temporary files
unlink("gadus_morhua_cube.nc")

# Final check of the cube structure
print(st_cube)


In [ ]:
?
# Calculate spatial autocorrelation (Moran's I) for the mean density
library(spdep)
library(sf)

# Convert stars object to sf for spatial analysis
sf_density <- st_as_sf(mean_density, as_points = TRUE)
sf_density <- sf_density[!is.na(sf_density[[1]]), ]

# Create neighbors and weights
nb <- dnearneigh(st_centroid(sf_density), 0, 5)
lw <- nb2listw(nb, style = "W", zero.policy = TRUE)

# Calculate Moran's I
moran_test <- moran.test(sf_density[[1]], lw, zero.policy = TRUE)
print(moran_test)

# Visualize spatial clusters
lisa <- localmoran(sf_density[[1]], lw, zero.policy = TRUE)
sf_density$lisa <- lisa[, 1]
plot(sf_density["lisa"], main = "Local Moran's I of Gadus morhua density")

# Perform Getis-Ord Gi* analysis for hot spot detection
gi_star <- localG(sf_density[[1]], lw)
sf_density$gi_star <- gi_star
plot(sf_density["gi_star"], main = "Getis-Ord Gi* Hot Spots of Gadus morhua")

# Save results to shapefile
st_write(sf_density, "gadus_morhua_spatial_stats.shp", delete_dsn = TRUE)

# Calculate spatial clustering significance
p_values <- pnorm(-abs(gi_star))
sf_density$p_val <- p_values
sf_density$cluster_type <- ifelse(sf_density$gi_star > 1.96, "Hot Spot", 
                                  ifelse(sf_density$gi_star < -1.96, "Cold Spot", "Not Significant"))

# Plot significant clusters
plot(sf_density["cluster_type"], main = "Significant Hot/Cold Spots (p < 0.05)")

# Export the final spatial analysis results
st_write(sf_density, "gadus_morhua_clusters.gpkg", delete_dsn = TRUE)

# Print summary of identified clusters
print(table(sf_density$cluster_type))

# Clean up temporary files
unlink("gadus_morhua_spatial_stats.shp")
unlink("gadus_morhua_spatial_stats.dbf")
unlink("gadus_morhua_spatial_stats.shx")
unlink("gadus_morhua_spatial_stats.prj")

# Final cleanup of workspace
rm(list = ls())
gc()

# Final check of memory and environment
print("Spatial analysis complete. Workspace cleared.")

# Final check of the environment
sessionInfo()
